# Ground sourced Biomass Mapping by random forest models in GEE

In [1]:
# import the libraries
import ee
import pandas as pd
import os
import numpy as np
import random
from random import sample
import itertools 
import geopandas as gpd
from sklearn.metrics import r2_score
from termcolor import colored # this is allocate colour and fonts type for the print title and text
from IPython.display import display, HTML

In [2]:
#set the working directory of local drive for Grid search result table loading
# os.getcwd()
os.chdir('/Users/LeonidMoore/Desktop/BIOMASS')

In [3]:
# initialize the earth engine API
ee.Initialize()

## STEP 1 Data preperation and objects definition

In [4]:
# check the parameter from grid search
# load the grid searh table from R
parameterTable = pd.read_csv('GroundSourcedModel/GridSearchResults/Ground_Sourced_Biomass_Potential_Modeling_Grid_Search_Seed_0.csv', float_precision='round_trip')
# show the structure by head function
print(colored('The head of the table: \n', 'blue', attrs=['bold']))
parameterTable.head()

The head of the table: 



,Unnamed: 0,Mean_R2,StDev_R2,ModelName,numberOfTrees,variablesPerSplit,minLeafPopulation,bagFraction,maxNodes,type
0,0,0.376014,0.033265,GridSeach_Model_15_14_80,200,15,14,0.632,80,Classifier.smileRandomForest


In [5]:
# load the hansen forest cover, and transform to 0-1
hansenTreeCover = ee.Image("projects/crowtherlab/Composite/CrowtherLab_Composite_30ArcSec").select('HansenEtAl_TreeCover_Year2010').divide(100).rename('PresentTreeCover')
# read the composite
compositeImage = ee.Image("users/leonidmoore/ForestBiomass/20200915_Forest_Biomass_Predictors_Image").addBands(hansenTreeCover)

# show the band names of the composite image 
print('Composite Band Names:',compositeImage.bandNames().getInfo())

Composite Band Names: ['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvCloudCover_MODCF_interannualSD', 'EarthEnvCloudCover_MODCF_intraannualSD', 'EarthEnvCloudCover_MODCF_meanannual', 'EarthEnvTopoMed_AspectCosine', 'EarthEnvTopoMed_AspectSine', 'EarthE

In [6]:
# define the boundary geography reference
unboundedGeo = ee.Geometry.Polygon([-180, 88, 0, 88, 180, 88, 180, -88, 0, -88, -180, -88], None, False)

In [7]:
# define the list of predictors
propertyOfInterest = ['Aridity_Index',
                      'CHELSA_Annual_Mean_Temperature',
                      'CHELSA_Annual_Precipitation',
                      'CHELSA_Isothermality',
                      'CHELSA_Max_Temperature_of_Warmest_Month',
                      'CHELSA_Mean_Diurnal_Range',
                      'CHELSA_Mean_Temperature_of_Coldest_Quarter',
                      'CHELSA_Mean_Temperature_of_Driest_Quarter',
                      'CHELSA_Mean_Temperature_of_Warmest_Quarter',
                      'CHELSA_Mean_Temperature_of_Wettest_Quarter',
                      'CHELSA_Min_Temperature_of_Coldest_Month',
                      'CHELSA_Precipitation_Seasonality',
                      'CHELSA_Precipitation_of_Coldest_Quarter',
                      'CHELSA_Precipitation_of_Driest_Month',
                      'CHELSA_Precipitation_of_Driest_Quarter',
                      'CHELSA_Precipitation_of_Warmest_Quarter',
                      'CHELSA_Precipitation_of_Wettest_Month',
                      'CHELSA_Precipitation_of_Wettest_Quarter',
                      'CHELSA_Temperature_Annual_Range',
                      'CHELSA_Temperature_Seasonality',
                      'Depth_to_Water_Table',
                      'EarthEnvTopoMed_Eastness',
                      'EarthEnvTopoMed_Elevation',
                      'EarthEnvTopoMed_Northness',
                      'EarthEnvTopoMed_ProfileCurvature',
                      'EarthEnvTopoMed_Roughness',
                      'EarthEnvTopoMed_Slope',
                      'SG_Absolute_depth_to_bedrock',
                      'WorldClim2_SolarRadiation_AnnualMean',
                      'WorldClim2_WindSpeed_AnnualMean',
                      'EarthEnvCloudCover_MODCF_interannualSD',
                      'EarthEnvCloudCover_MODCF_intraannualSD',
                      'EarthEnvCloudCover_MODCF_meanannual',
                      'EarthEnvTopoMed_AspectCosine',
                      'EarthEnvTopoMed_AspectSine',
                      'SG_Clay_Content_0_100cm',
                      'SG_Coarse_fragments_0_100cm',
                      'SG_Sand_Content_0_100cm',
                      'SG_Silt_Content_0_100cm',
                      'SG_Soil_pH_H2O_0_100cm']#
print(propertyOfInterest)

['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvTopoMed_Eastness', 'EarthEnvTopoMed_Elevation', 'EarthEnvTopoMed_Northness', 'EarthEnvTopoMed_ProfileCurvature', 'EarthEnvTopoMed_Roughness', 'EarthEnvTopoMed_Slope', 'SG_Absolute_depth_to_bedrock', 'World

### STEP 2 Mapping Potential Biomass on Google earth engine

#### 2.1 Machine learning mapping

In [1]:
# define a loop through the seed list
seedList = np.arange(0, 100, 1).tolist()
print(colored('The models are:', 'blue', attrs=['bold']),seedList)
print(colored('Model is running:\nWith paramter sets:', 'blue', attrs=['bold']))
# for seed in seedList: range(0,len(seedList))
for seed in seedList:
    # load the points data with the covariates
    trainTable = ee.FeatureCollection('users/leonidmoore/ForestBiomass/GroundSourcedModel/TrainTables/Ground_Sourced_data_Potential_Grid_Subsampled_Train_Table_seed_'+str(seed))
    # print(trainTable.size().getInfo())
    parameterTable = pd.read_csv('GroundSourcedModel/GridSearchResults/Ground_Sourced_Biomass_Potential_Modeling_Grid_Search_Seed_'+str(seed)+'.csv', float_precision='round_trip')
    # not recomend to run the code below
    # print(parameterTable.head())
    # extract the paramters
    variablesPerSplitVal = int(parameterTable['variablesPerSplit'].iat[0]) # mtry
    minLeafPopulationVal = int(parameterTable['minLeafPopulation'].iat[0]) # minrow
    maxNodesVal = int(parameterTable['maxNodes'].iat[0]) # mac depth
    print('seed',seed,variablesPerSplitVal,minLeafPopulationVal,maxNodesVal)
    # define the random forest classifier
    rfClassifier = ee.Classifier.smileRandomForest(numberOfTrees = 200,
                                                   variablesPerSplit = variablesPerSplitVal, # mtry
                                                   minLeafPopulation = minLeafPopulationVal, # minrow
                                                   maxNodes = maxNodesVal, # max depth
                                                   bagFraction = 0.632,
                                                   seed = seed).setOutputMode('REGRESSION')
    trainedClassifier = rfClassifier.train(features = trainTable,
                                           classProperty = 'lgBD',
                                           inputProperties = propertyOfInterest)
    # execute the prediction to generate the map
    potentialCarbonDensityMap = compositeImage.classify(trainedClassifier)
    # print(predictedWoodDensityMap.getInfo())
    predictionExport = ee.batch.Export.image.toAsset(image = potentialCarbonDensityMap.exp().subtract(1),
                                                     description = '20220208_GS_Potential_Biomass_Density_Map_To_Asset_'+str(seed),
                                                     assetId = 'users/leonidmoore/ForestBiomass/GroundSourcedModel/PredictedMaps/Predicted_Potential_Biomass_Map_Seed_'+str(seed),
                                                     region = unboundedGeo,
                                                     crs = 'EPSG:4326',
                                                     crsTransform = [0.008333333333333333,0,-180,0,-0.008333333333333333,90],
                                                     maxPixels = 1e13)

    # print(predictionExportAsset)
    # start the export task
    predictionExport.start()
    # show the task status
    predictionExport.status()

NameError: name 'np' is not defined